# LeetCode 1442 — Count Triplets That Can Form Two Arrays of Equal XOR bằng Python

Link đề: https://leetcode.com/problems/count-triplets-that-can-form-two-arrays-of-equal-xor/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Math, Bit Manipulation, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def countTriplets(self, arr: list[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Nắm phép **XOR** (`^`) và 3 tính chất vàng: `x ^ x = 0`, `x ^ 0 = x`, giao hoán & kết hợp.
2. Dùng **prefix XOR** để tính XOR của một đoạn bất kỳ trong `O(1)`: `xor(i..k) = P[k+1] ^ P[i]`.
3. Phát hiện bước đột phá: `a == b` ⇔ `a ^ b == 0` ⇔ **XOR cả đoạn `i..k` bằng 0** — khi đó **mọi** `j` trong `(i, k]` đều thoả ⇒ cộng `k − i`.
4. Đi từ brute force `O(n⁴)` → `O(n³)` → `O(n²)` → `O(n)` với **dict đếm số lần + tổng chỉ số**.
5. Tránh lỗi: nhầm `j < k` với `j ≤ k`, chỉ cộng 1 thay vì `k − i`, quên `P[0] = 0`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr`. Chọn ba chỉ số `i, j, k` với `0 ≤ i < j ≤ k < len(arr)`. Định nghĩa
- `a = arr[i] ^ arr[i+1] ^ … ^ arr[j−1]` (đoạn bên trái, kết thúc **trước** `j`),
- `b = arr[j] ^ arr[j+1] ^ … ^ arr[k]` (đoạn bên phải, bắt đầu **tại** `j`).

Đếm số bộ ba `(i, j, k)` sao cho `a == b`. (`^` là phép XOR bit.)

Chú ý `j ≤ k` nên đoạn phải có thể chỉ gồm 1 phần tử; `i < j` nên đoạn trái cũng có ít nhất 1 phần tử.

**Ví dụ 1:** `arr = [2,3,1,6,7]` → `4`; các bộ ba là `(0,1,2), (0,2,2), (2,3,4), (2,4,4)`.

**Ví dụ 2:** `arr = [1,1,1,1,1]` → `10`.

**Ràng buộc:** `1 <= arr.length <= 300`, `1 <= arr[i] <= 10^8`.

In [1]:
import random
import time
from collections import defaultdict
from functools import reduce
from operator import xor
from typing import List

examples = [([2, 3, 1, 6, 7], 4), ([1, 1, 1, 1, 1], 10)]
for arr, want in examples:
    print(f"arr={arr} -> kỳ vọng {want}")

arr=[2, 3, 1, 6, 7] -> kỳ vọng 4
arr=[1, 1, 1, 1, 1] -> kỳ vọng 10


## 3. Kiến thức nền

### 3.1 Phép XOR `^`

XOR so từng bit: hai bit **khác nhau** cho 1, **giống nhau** cho 0. Trong Python `^` là XOR (không phải luỹ thừa — luỹ thừa là `**`).

In [2]:
a, b = 6, 3
print(f"{a:04b} ^ {b:04b} = {a ^ b:04b} ({a ^ b})")
print("2 ^ 3 =", 2 ^ 3, "| 2 ** 3 =", 2 ** 3)

0110 ^ 0011 = 0101 (5)
2 ^ 3 = 1 | 2 ** 3 = 8


### 3.2 Ba tính chất vàng của XOR

1. `x ^ x = 0` (tự triệt tiêu).
2. `x ^ 0 = x` (0 là phần tử trung hoà).
3. Giao hoán, kết hợp: thứ tự XOR không quan trọng.

Hệ quả cực mạnh: **`a == b` ⇔ `a ^ b == 0`**. (Nếu `a == b` thì `a ^ b = a ^ a = 0`; ngược lại nếu `a ^ b = 0` thì XOR hai vế với `b`: `a = b`.)

In [3]:
for x in [0, 5, 12345]:
    print(x ^ x, x ^ 0 == x)
print((2 ^ 3) ^ 1 == 2 ^ (3 ^ 1) == 1 ^ 2 ^ 3)
for a, b in [(4, 4), (4, 5), (7, 7)]:
    print(a, b, "a==b:", a == b, "| a^b==0:", (a ^ b) == 0)

0 True
0 True
0 True
True
4 4 a==b: True | a^b==0: True
4 5 a==b: False | a^b==0: False
7 7 a==b: True | a^b==0: True


### 3.3 XOR của cả một dãy: `functools.reduce`

`reduce(xor, [x1, x2, x3], 0)` = `0 ^ x1 ^ x2 ^ x3`. Tiện cho brute force.

In [4]:
print(reduce(xor, [2, 3, 1], 0), "=", 2 ^ 3 ^ 1)
print(reduce(xor, [], 0), "<- dãy rỗng cho 0 nhờ giá trị khởi đầu")

0 = 0
0 <- dãy rỗng cho 0 nhờ giá trị khởi đầu


### 3.4 Prefix XOR

Giống prefix sum: `P[0] = 0`, `P[t+1] = P[t] ^ arr[t]`. Khi đó `P[t]` = XOR của `arr[0..t−1]` và

`xor(arr[i..k]) = P[k+1] ^ P[i]` — vì phần chung `arr[0..i−1]` xuất hiện hai lần và tự triệt tiêu.

In [5]:
arr = [2, 3, 1, 6, 7]
P = [0]
for x in arr:
    P.append(P[-1] ^ x)
print("P =", P)
i, k = 1, 3
print("xor(arr[1..3]) =", P[k + 1] ^ P[i], "| trực tiếp:", 3 ^ 1 ^ 6)

P = [0, 2, 1, 0, 6, 1]
xor(arr[1..3]) = 4 | trực tiếp: 4


### 3.5 `defaultdict(int)` để đếm

`defaultdict(int)` tự tạo giá trị 0 cho khoá chưa có — đếm số lần xuất hiện hoặc cộng dồn mà không cần `if key in d`.

In [6]:
cnt = defaultdict(int)
total = defaultdict(int)
for idx, v in enumerate([0, 2, 1, 0, 6, 1]):
    cnt[v] += 1
    total[v] += idx
print(dict(cnt))
print(dict(total))

{0: 2, 2: 1, 1: 2, 6: 1}
{0: 3, 2: 1, 1: 7, 6: 4}


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1 `arr = [2,3,1,6,7]`, bộ `(0,1,2)`: `a = arr[0] = 2`, `b = arr[1]^arr[2] = 3^1 = 2` ⇒ bằng nhau ✔. Bộ `(0,2,2)`: `a = 2^3 = 1`, `b = arr[2] = 1` ✔.

Để ý: cả hai bộ có cùng `i = 0, k = 2`, chỉ khác `j`. XOR cả đoạn `arr[0..2] = 2^3^1 = 0`. Đó không phải trùng hợp:

- `a ^ b` = XOR của **cả đoạn** `arr[i..k]` (đoạn trái nối đoạn phải).
- `a == b` ⇔ `a ^ b == 0` ⇔ `xor(arr[i..k]) == 0`.
- Điều kiện **không phụ thuộc `j`**! Nên khi đoạn `i..k` có XOR bằng 0, **mọi** `j ∈ {i+1, …, k}` đều đúng ⇒ có `k − i` bộ ba.

Kiểm tra: đoạn `[0..2]` XOR = 0 → `k − i = 2` bộ; đoạn `[2..4] = 1^6^7 = 0` → 2 bộ. Tổng 4 ✔.

In [7]:
arr = [2, 3, 1, 6, 7]
n = len(arr)
for i in range(n):
    for k in range(i + 1, n):
        seg = arr[i:k + 1]
        if reduce(xor, seg, 0) == 0:
            print(f"đoạn i={i}..k={k} {seg} có XOR = 0 -> cộng k - i = {k - i} bộ ba (j = {list(range(i+1, k+1))})")

đoạn i=0..k=2 [2, 3, 1] có XOR = 0 -> cộng k - i = 2 bộ ba (j = [1, 2])
đoạn i=2..k=4 [1, 6, 7] có XOR = 0 -> cộng k - i = 2 bộ ba (j = [3, 4])


## 5. Cách 1 — Brute Force / baseline

### 5.1 Ngây thơ `O(n⁴)`

Duyệt mọi `(i, j, k)`, tính `a` và `b` bằng vòng lặp (hoặc `reduce`) — ba vòng chọn chỉ số và một vòng tính XOR.

In [8]:
def count_brute4(arr: List[int]) -> int:
    n, cnt = len(arr), 0
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j, n):
                a = reduce(xor, arr[i:j], 0)
                b = reduce(xor, arr[j:k + 1], 0)
                if a == b:
                    cnt += 1
    return cnt

for arr, want in examples:
    print(count_brute4(arr), "kỳ vọng", want)

4 kỳ vọng 4
10 kỳ vọng 10


### 5.2 Cải tiến `O(n³)` bằng prefix XOR

Tính `a = P[j] ^ P[i]`, `b = P[k+1] ^ P[j]` trong `O(1)`. Còn ba vòng lặp lồng nhau.

In [9]:
def count_brute3(arr: List[int]) -> int:
    n = len(arr)
    P = [0] * (n + 1)
    for t, x in enumerate(arr):
        P[t + 1] = P[t] ^ x
    cnt = 0
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j, n):
                if P[j] ^ P[i] == P[k + 1] ^ P[j]:
                    cnt += 1
    return cnt

for arr, want in examples:
    print(count_brute3(arr), "kỳ vọng", want)

4 kỳ vọng 4
10 kỳ vọng 10


**Độ phức tạp:** `O(n⁴)` và `O(n³)`; với `n = 300`, `n³/6 ≈ 4,5·10⁶` phép — Python mất cỡ nửa giây (vẫn qua được nhưng không đẹp), còn `O(n⁴)` thì quá chậm.

**Nút thắt:** vòng `j` là thừa — điều kiện thực ra không phụ thuộc `j` (mục 4).

In [10]:
rng = random.Random(0)
arr150 = [rng.randint(1, 10**8) for _ in range(150)]
t = time.perf_counter()
r3 = count_brute3(arr150)
print(f"O(n^3) với n=150: {r3} ({time.perf_counter()-t:.2f}s) — n=300 sẽ chậm gấp ~8 lần")

O(n^3) với n=150: 0 (0.03s) — n=300 sẽ chậm gấp ~8 lần


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** `a == b` ⇔ `xor(arr[i..k]) == 0` ⇔ `P[i] == P[k+1]`. Khi đúng, đóng góp **`k − i`** bộ ba.

**Insight 2 (đổi biến):** đặt `x = i`, `y = k + 1` (hai vị trí trong mảng prefix `P[0..n]`, `x < y`, `y − x ≥ 2`). Đóng góp `k − i = y − x − 1`. Nếu `y − x = 1` thì đóng góp 0, nên có thể bỏ điều kiện `y − x ≥ 2` mà không sai.

⇒ Đáp án = **tổng `(y − x − 1)` trên mọi cặp `x < y` có `P[x] == P[y]`**.

**Insight 3 (gộp theo nhóm):** với mỗi `y`, gọi `c` = số chỉ số `x < y` có `P[x] = P[y]`, và `s` = tổng các `x` đó. Tổng đóng góp của `y` là

`Σ (y − 1 − x) = c·(y − 1) − s`.

Giữ hai dict `cnt[v]` và `sumidx[v]` khi quét `y` từ trái sang phải ⇒ `O(n)`.

**Quy tắc nhớ:** "nếu prefix XOR lặp lại giá trị cũ thì đoạn giữa có XOR 0 ⇒ cộng độ dài trừ 1".

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `O(n²)`: duyệt `(i, k)`, cộng `k − i` khi XOR đoạn bằng 0

**Giải thích:** cố định `i`, kéo `k` sang phải và giữ `cur = arr[i] ^ … ^ arr[k]` chạy dần (không cần mảng prefix). Nếu `cur == 0` (và `k > i`, tự động vì `arr[i] ≥ 1`) thì cộng `k − i`.

In [11]:
def count_n2(arr: List[int]) -> int:
    n, ans = len(arr), 0
    for i in range(n):
        cur = 0
        for k in range(i, n):
            cur ^= arr[k]
            if cur == 0:
                ans += k - i
    return ans

for arr, want in examples:
    print(count_n2(arr), "kỳ vọng", want)

4 kỳ vọng 4
10 kỳ vọng 10


Chú ý: khi `k == i`, `cur = arr[i] ≥ 1 ≠ 0`, nhưng kể cả nếu bằng 0 thì `k − i = 0` cũng không cộng gì — an toàn.

**Code nộp LeetCode (cách 2):**

In [12]:
class SolutionN2:
    def countTriplets(self, arr: List[int]) -> int:
        n, ans = len(arr), 0
        for i in range(n):
            cur = 0
            for k in range(i, n):
                cur ^= arr[k]
                if cur == 0:
                    ans += k - i
        return ans

print(SolutionN2().countTriplets([2, 3, 1, 6, 7]))

4


**Độ phức tạp:** Time `O(n²)` (≈ 45 000 bước với `n = 300`), Space `O(1)`.

### 7.2 Cách 3 — `O(n²)` theo cặp prefix bằng nhau (cầu nối sang `O(n)`)

Viết lại theo ngôn ngữ prefix: duyệt mọi cặp `x < y` trong `P`, nếu `P[x] == P[y]` cộng `y − x − 1`. Cùng độ phức tạp với cách 2, nhưng thể hiện rõ công thức mà cách 4 sẽ gộp lại.

In [13]:
def count_prefix_pairs(arr: List[int]) -> int:
    P = [0]
    for x in arr:
        P.append(P[-1] ^ x)
    ans = 0
    for x in range(len(P)):
        for y in range(x + 1, len(P)):
            if P[x] == P[y]:
                ans += y - x - 1
    return ans

for arr, want in examples:
    print(count_prefix_pairs(arr), "kỳ vọng", want)

4 kỳ vọng 4
10 kỳ vọng 10


### 7.3 Cách 4 — `O(n)`: prefix XOR + dict (đếm, tổng chỉ số)

**Giải thích từng đoạn:**
1. Khởi tạo `cnt = {0: 1}`, `sumidx = {0: 0}` — đại diện cho `P[0] = 0` ở chỉ số `x = 0`.
2. Quét `y = 1..n`, cập nhật `pre ^= arr[y−1]` (chính là `P[y]`).
3. Cộng `cnt[pre] * (y − 1) − sumidx[pre]` vào đáp án (công thức insight 3).
4. Ghi nhận `y` vào nhóm `pre`: `cnt[pre] += 1`, `sumidx[pre] += y`.

Thứ tự "cộng trước, ghi nhận sau" đảm bảo chỉ đếm các `x < y`.

In [14]:
def count_linear(arr: List[int]) -> int:
    cnt = defaultdict(int)
    sumidx = defaultdict(int)
    cnt[0], sumidx[0] = 1, 0          # P[0] = 0 tại chỉ số 0
    pre = ans = 0
    for y in range(1, len(arr) + 1):
        pre ^= arr[y - 1]             # pre = P[y]
        ans += cnt[pre] * (y - 1) - sumidx[pre]
        cnt[pre] += 1
        sumidx[pre] += y
    return ans

for arr, want in examples:
    print(count_linear(arr), "kỳ vọng", want)

4 kỳ vọng 4
10 kỳ vọng 10


**Code nộp LeetCode (cách 4)** — xem mục 9. **Độ phức tạp:** Time `O(n)`, Space `O(n)` (tối đa `n + 1` giá trị prefix khác nhau).

So sánh tốc độ với `n = 300` (giới hạn đề) và với `n = 10^5` (ngoài đề, chỉ để thấy sự khác biệt `O(n)`):

In [15]:
rng = random.Random(1)
arr300 = [rng.randint(1, 8) for _ in range(300)]   # giá trị nhỏ để có nhiều đoạn XOR 0
for name, f in [("O(n^2)", count_n2), ("prefix pairs", count_prefix_pairs), ("O(n) dict", count_linear)]:
    t = time.perf_counter()
    r = f(arr300)
    print(f"{name:13} n=300 -> {r} ({time.perf_counter()-t:.4f}s)")
big = [rng.randint(1, 8) for _ in range(10**5)]
t = time.perf_counter()
print("O(n) dict, n=10^5 ->", count_linear(big), f"({time.perf_counter()-t:.3f}s)")

O(n^2)        n=300 -> 278837 (0.0015s)
prefix pairs  n=300 -> 278837 (0.0013s)
O(n) dict     n=300 -> 278837 (0.0001s)
O(n) dict, n=10^5 -> 10416224955756 (0.018s)


## 8. TRACE từng bước

Trace cách 4 trên ví dụ 1 `arr = [2,3,1,6,7]`. Mảng prefix `P = [0,2,1,0,6,1]`.

| y | arr[y−1] | pre = P[y] | cnt[pre] trước | sumidx[pre] trước | cộng = cnt·(y−1) − sum | ans |
|---|---|---|---|---|---|---|
| 1 | 2 | 2 | 0 | 0 | 0 | 0 |
| 2 | 3 | 1 | 0 | 0 | 0 | 0 |
| 3 | 1 | 0 | 1 (x=0) | 0 | 1·2 − 0 = 2 | 2 |
| 4 | 6 | 6 | 0 | 0 | 0 | 2 |
| 5 | 7 | 1 | 1 (x=2) | 2 | 1·4 − 2 = 2 | 4 |

Hai lần cộng ứng với đoạn `arr[0..2]` (x=0, y=3) và `arr[2..4]` (x=2, y=5).

In [16]:
def trace_linear(arr):
    cnt, sumidx = defaultdict(int), defaultdict(int)
    cnt[0], sumidx[0] = 1, 0
    pre = ans = 0
    for y in range(1, len(arr) + 1):
        pre ^= arr[y - 1]
        c, s = cnt[pre], sumidx[pre]
        add = c * (y - 1) - s
        ans += add
        cnt[pre] += 1
        sumidx[pre] += y
        print(f"y={y} arr[y-1]={arr[y-1]} pre={pre} | cnt={c} sum={s} -> cộng {c}*{y-1}-{s}={add} | ans={ans}")
    return ans

trace_linear([2, 3, 1, 6, 7])

y=1 arr[y-1]=2 pre=2 | cnt=0 sum=0 -> cộng 0*0-0=0 | ans=0
y=2 arr[y-1]=3 pre=1 | cnt=0 sum=0 -> cộng 0*1-0=0 | ans=0
y=3 arr[y-1]=1 pre=0 | cnt=1 sum=0 -> cộng 1*2-0=2 | ans=2
y=4 arr[y-1]=6 pre=6 | cnt=0 sum=0 -> cộng 0*3-0=0 | ans=2
y=5 arr[y-1]=7 pre=1 | cnt=1 sum=2 -> cộng 1*4-2=2 | ans=4


4

Trace thêm ví dụ 2 `[1,1,1,1,1]`: prefix luân phiên `0,1,0,1,0,1` nên mỗi giá trị lặp lại nhiều lần — xem công thức gộp hoạt động:

In [17]:
trace_linear([1, 1, 1, 1, 1])

y=1 arr[y-1]=1 pre=1 | cnt=0 sum=0 -> cộng 0*0-0=0 | ans=0
y=2 arr[y-1]=1 pre=0 | cnt=1 sum=0 -> cộng 1*1-0=1 | ans=1
y=3 arr[y-1]=1 pre=1 | cnt=1 sum=1 -> cộng 1*2-1=1 | ans=2
y=4 arr[y-1]=1 pre=0 | cnt=2 sum=2 -> cộng 2*3-2=4 | ans=6
y=5 arr[y-1]=1 pre=1 | cnt=2 sum=4 -> cộng 2*4-4=4 | ans=10


10

## 9. Code nộp LeetCode

Lời giải chính (cách 4, `O(n)`), đúng chữ ký `countTriplets(self, arr)`.

In [18]:
from collections import defaultdict
from typing import List

class Solution:
    def countTriplets(self, arr: List[int]) -> int:
        cnt = defaultdict(int)
        sumidx = defaultdict(int)
        cnt[0] = 1                    # P[0] = 0 ở chỉ số 0 (sumidx[0] = 0)
        pre = ans = 0
        for y in range(1, len(arr) + 1):
            pre ^= arr[y - 1]
            ans += cnt[pre] * (y - 1) - sumidx[pre]
            cnt[pre] += 1
            sumidx[pre] += y
        return ans

for arr, want in examples:
    assert Solution().countTriplets(arr) == want
print("Solution: ví dụ đề OK")

Solution: ví dụ đề OK


## 10. Vì sao thuật toán đúng

1. **Tương đương điều kiện:** `a ^ b = (arr[i]^…^arr[j−1]) ^ (arr[j]^…^arr[k]) = xor(arr[i..k])`. Theo tính chất XOR, `a == b ⇔ a ^ b == 0`. Vậy bộ `(i, j, k)` hợp lệ ⇔ `xor(arr[i..k]) = 0` và `i < j ≤ k`.
2. **Đếm theo `(i, k)`:** với mỗi cặp `(i, k)` có `xor(arr[i..k]) = 0`, số `j` hợp lệ là `|{i+1, …, k}| = k − i`. Các cặp `(i, k)` khác nhau cho các bộ ba khác nhau ⇒ tổng không đếm trùng (cách 2 đúng).
3. **Đổi sang prefix:** `xor(arr[i..k]) = P[k+1] ^ P[i]`, bằng 0 ⇔ `P[i] = P[k+1]`. Với `x = i, y = k+1`, đóng góp `y − x − 1` (cách 3 đúng; các cặp `y = x + 1` đóng góp 0 nên không ảnh hưởng).
4. **Gộp:** khi xét `y`, dict chứa đúng các `x < y` (vì ghi nhận `y` **sau** khi cộng), nhóm theo giá trị `P[x]`. `Σ_{x∈nhóm} (y − 1 − x) = c·(y − 1) − s`. Cộng trên mọi `y` cho đúng tổng ở bước 3 (cách 4 đúng).

## 11. Phân tích Time / Space

Gọi `n = len(arr)` (`≤ 300`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1a. Brute `O(n⁴)` | `O(n⁴)` | `O(n)` (slice) | chỉ để hiểu đề |
| 1b. Brute + prefix | `O(n³)` | `O(n)` | ≈ 4,5·10⁶ phép với n=300 |
| 2. `(i,k)` + XOR chạy | `O(n²)` | `O(1)` | đủ nhanh cho đề |
| 3. Cặp prefix bằng nhau | `O(n²)` | `O(n)` | cầu nối tới công thức gộp |
| 4. Prefix + dict | `O(n)` | `O(n)` | tối ưu, mở rộng được cho n lớn |

Giá trị `arr[i] ≤ 10^8 < 2^27` nên mọi XOR vẫn là số nhỏ, phép `^` là `O(1)`.

## 12. Bộ test

Ví dụ đề, edge case (1 phần tử, 2 phần tử bằng/khác nhau, toàn bằng nhau, mảng lớn nhất), và đối chiếu ngẫu nhiên mọi cách với brute force (seed cố định). Giá trị nhỏ được chọn có chủ đích để có nhiều đoạn XOR bằng 0.

In [19]:
ALL = {
    "brute4": count_brute4,
    "brute3": count_brute3,
    "n2": count_n2,
    "prefix_pairs": count_prefix_pairs,
    "linear": count_linear,
    "Solution": lambda a: Solution().countTriplets(a),
}
edge = [
    ([5], 0),
    ([1, 1], 1),          # (0,1,1)
    ([1, 2], 0),
    ([7, 7, 7], 2),       # đoạn [0..1] và [1..2], mỗi đoạn 1 bộ
    ([1, 2, 3], 2),       # 1^2^3 = 0 -> k-i = 2
    ([4, 4, 4, 4], 1 + 1 + 1 + 3),
]
for arr, want in examples + edge:
    for name, f in ALL.items():
        got = f(arr)
        assert got == want, (name, arr, got, want)
print("Ví dụ đề + edge case: OK")

Ví dụ đề + edge case: OK


In [20]:
rng = random.Random(1442)
for _ in range(400):
    n = rng.randint(1, 14)
    hi = rng.choice([1, 3, 7, 10**8])
    arr = [rng.randint(1, hi) for _ in range(n)]
    want = count_brute4(arr)
    for name, f in ALL.items():
        assert f(arr) == want, (name, arr)
print("400 test ngẫu nhiên nhỏ: mọi cách khớp brute O(n^4)")

for _ in range(30):
    arr = [rng.randint(1, rng.choice([3, 15, 10**8])) for _ in range(300)]
    assert count_n2(arr) == count_prefix_pairs(arr) == count_linear(arr)
print("30 test n=300 (giới hạn đề): O(n^2) và O(n) khớp nhau")
print("Toàn 1, n=300:", count_linear([1] * 300))

400 test ngẫu nhiên nhỏ: mọi cách khớp brute O(n^4)


30 test n=300 (giới hạn đề): O(n^2) và O(n) khớp nhau
Toàn 1, n=300: 2250050


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đọc nhầm `j ≤ k` thành `j < k`

Đề cho `i < j ≤ k`: đoạn phải `arr[j..k]` có thể chỉ một phần tử. Viết `range(j + 1, n)` cho `k` sẽ bỏ sót.

In [21]:
def wrong_strict_k(arr):
    n, P = len(arr), [0]
    for x in arr:
        P.append(P[-1] ^ x)
    cnt = 0
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):       # SAI: phải là range(j, n)
                if P[j] ^ P[i] == P[k + 1] ^ P[j]:
                    cnt += 1
    return cnt

for arr in ([1, 1], [2, 3, 1, 6, 7]):
    print(f"{arr}: Sai {wrong_strict_k(arr)} | Đúng {count_linear(arr)}")

[1, 1]: Sai 0 | Đúng 1
[2, 3, 1, 6, 7]: Sai 2 | Đúng 4


**Sửa:** `for k in range(j, n)`.

### Lỗi 2 — Tìm được đoạn XOR 0 nhưng chỉ cộng 1

Mỗi đoạn `i..k` có XOR 0 cho **`k − i`** bộ ba (mọi `j` ở giữa), không phải 1.

In [22]:
def wrong_plus_one(arr):
    n, ans = len(arr), 0
    for i in range(n):
        cur = 0
        for k in range(i, n):
            cur ^= arr[k]
            if cur == 0:
                ans += 1           # SAI
    return ans

print("Sai :", wrong_plus_one([1, 1, 1, 1, 1]), "| Đúng:", count_linear([1, 1, 1, 1, 1]))
print("Sai :", wrong_plus_one([2, 3, 1, 6, 7]), "| Đúng:", count_linear([2, 3, 1, 6, 7]))

Sai : 6 | Đúng: 10
Sai : 2 | Đúng: 4


**Sửa:** `ans += k - i`.

### Lỗi 3 — Quên `P[0] = 0` (khởi tạo dict rỗng)

Thiếu "prefix rỗng" thì mọi đoạn **bắt đầu tại 0** bị bỏ sót.

In [23]:
def wrong_no_p0(arr):
    cnt, sumidx = defaultdict(int), defaultdict(int)   # SAI: thiếu cnt[0] = 1
    pre = ans = 0
    for y in range(1, len(arr) + 1):
        pre ^= arr[y - 1]
        ans += cnt[pre] * (y - 1) - sumidx[pre]
        cnt[pre] += 1
        sumidx[pre] += y
    return ans

for arr in ([2, 3, 1, 6, 7], [1, 1]):
    print(f"{arr}: Sai {wrong_no_p0(arr)} | Đúng {count_linear(arr)}")

[2, 3, 1, 6, 7]: Sai 2 | Đúng 4
[1, 1]: Sai 0 | Đúng 1


**Sửa:** `cnt[0] = 1` (và `sumidx[0] = 0`) trước vòng lặp.

### Lỗi 4 — Ghi nhận `y` vào dict **trước** khi cộng

Nếu cập nhật `cnt[pre]`, `sumidx[pre]` trước, `y` sẽ tự ghép với chính nó: cộng thêm `(y − 1) − y = −1` mỗi bước ⇒ kết quả sai (có thể âm).

In [24]:
def wrong_order(arr):
    cnt, sumidx = defaultdict(int), defaultdict(int)
    cnt[0] = 1
    pre = ans = 0
    for y in range(1, len(arr) + 1):
        pre ^= arr[y - 1]
        cnt[pre] += 1              # SAI: ghi nhận trước
        sumidx[pre] += y
        ans += cnt[pre] * (y - 1) - sumidx[pre]
    return ans

for arr in ([2, 3, 1, 6, 7], [5]):
    print(f"{arr}: Sai {wrong_order(arr)} | Đúng {count_linear(arr)}")

[2, 3, 1, 6, 7]: Sai -1 | Đúng 4
[5]: Sai -1 | Đúng 0


**Sửa:** "cộng trước, ghi nhận sau".

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy **XOR của đoạn con** ⇒ nghĩ ngay đến **prefix XOR** (`P[k+1] ^ P[i]`).
- Thấy **"hai đại lượng XOR bằng nhau"** ⇒ đổi thành **"XOR của chúng bằng 0"**.
- "Đếm cặp `x < y` có `P[x] == P[y]`" ⇒ **hashmap đếm tần suất** quét một lượt.
- Đóng góp phụ thuộc vào **khoảng cách** `y − x` ⇒ giữ thêm **tổng chỉ số** để gộp (`c·y − s`).
- Ràng buộc `n ≤ 300` cho phép `O(n²)` hoặc thậm chí `O(n³)` — trong phỏng vấn hãy nói ra lời giải `O(n²)` trước rồi tối ưu.

## 15. Biến thể của bài

1. **Đếm subarray có XOR = K** (dạng "Subarray Sum Equals K" cho XOR): dict đếm `P[x] = P[y] ^ K` (bài tập 4).
2. **Đếm bộ ba với `a ^ b == K`:** tương đương đoạn `i..k` có XOR = K, có trọng số `k − i` (bài tập 5).
3. **XOR Queries of a Subarray** (LeetCode 1310): trả lời nhiều truy vấn XOR đoạn bằng prefix.
4. **Tổng thay XOR:** "chia đoạn thành hai phần có tổng bằng nhau" — không còn tính chất `a ^ a = 0`, phải dùng `a − b = 0` ⇒ `2·S(i..j−1) = S(i..k)`, phụ thuộc `j`, khó hơn.
5. **Longest subarray with XOR 0:** lưu chỉ số **xuất hiện đầu tiên** của mỗi prefix thay vì đếm.

In [25]:
def longest_zero_xor(arr):
    first = {0: 0}
    pre = best = 0
    for y in range(1, len(arr) + 1):
        pre ^= arr[y - 1]
        if pre in first:
            best = max(best, y - first[pre])
        else:
            first[pre] = y
    return best

print(longest_zero_xor([2, 3, 1, 6, 7]), longest_zero_xor([1, 2, 4]))

3 0


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. Brute | Mọi `(i,j,k)`, tính XOR bằng vòng lặp | `O(n⁴)` | `O(n)` | Kiểm chứng `n` rất nhỏ |
| 1b. Brute + prefix | Mọi `(i,j,k)`, XOR đoạn `O(1)` | `O(n³)` | `O(n)` | Khi chưa thấy insight |
| 2. `(i,k)` + XOR chạy | XOR đoạn = 0 ⇒ cộng `k − i` | `O(n²)` | `O(1)` | Đủ cho đề, dễ viết |
| 3. Cặp prefix | `P[x]==P[y]` ⇒ cộng `y − x − 1` | `O(n²)` | `O(n)` | Bước trung gian để hiểu cách 4 |
| 4. Prefix + dict | Gộp `c·(y−1) − s` | `O(n)` | `O(n)` | Tối ưu, `n` lớn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output):** `countTriplets([7,11,12,9,5,2,7,17,22])` bằng bao nhiêu?

**Bài 2 (tính tay):** viết mảng prefix XOR `P` (độ dài `n + 1`) của `arr = [2,3,1,6,7]` và chỉ ra các cặp `(x, y)` có `P[x] == P[y]`.

**Bài 3 (điền TODO):** hoàn thiện `count_todo` (cách 4).

**Bài 4 (tự viết lại):** viết `count_xor_k(arr, K)` đếm số subarray (độ dài ≥ 1) có XOR đúng bằng `K`, `O(n)`.

**Bài 5 (mini challenge):** viết `count_triplets_k(arr, K)` đếm số bộ `(i, j, k)` với `i < j ≤ k` sao cho `a ^ b == K` (đề gốc là `K = 0`), `O(n)`.

In [26]:
# Bài 3 — điền TODO (chỉ định nghĩa, không gọi)
def count_todo(arr):
    cnt, sumidx = defaultdict(int), defaultdict(int)
    # TODO: khởi tạo nhóm của P[0] = 0
    pre = ans = 0
    for y in range(1, len(arr) + 1):
        pre ^= arr[y - 1]
        # TODO: cộng đóng góp của mọi x < y có P[x] == pre
        # TODO: ghi nhận y vào nhóm pre
    return ans

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Tìm các đoạn có XOR bằng 0. (2) Dùng `count_n2` trong đầu hoặc tính prefix. (3) Mỗi đoạn `i..k` có XOR 0 cộng `k − i`.

**Bài 2:** (1) `P[0] = 0`, `P[t+1] = P[t] ^ arr[t]`. (2) `P = [0,2,1,0,6,1]`. (3) Các cặp bằng nhau: `(0,3)` và `(2,5)`.

**Bài 3:** (1) `cnt[0] = 1`. (2) `ans += cnt[pre] * (y - 1) - sumidx[pre]`. (3) `cnt[pre] += 1; sumidx[pre] += y` — đặt **sau** dòng cộng.

**Bài 4:** (1) Subarray `x..y−1` có XOR `P[y] ^ P[x]`. (2) Cần `P[x] = P[y] ^ K`. (3) Dict đếm `cnt`, khởi tạo `cnt[0] = 1`, cộng `cnt[pre ^ K]` trước khi ghi nhận.

**Bài 5:** (1) `a ^ b = xor(arr[i..k])` vẫn đúng. (2) Cần `P[x] = P[y] ^ K`, đóng góp `y − x − 1`. (3) Tra `cnt[pre ^ K]` và `sumidx[pre ^ K]`. Lưu ý khi `K ≠ 0` thì cặp `y = x + 1` (đoạn 1 phần tử) đóng góp `0` — vẫn đúng vì không có `j` nào.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1
ans1 = count_linear([7, 11, 12, 9, 5, 2, 7, 17, 22])
print("Bài 1:", ans1)
assert ans1 == count_brute3([7, 11, 12, 9, 5, 2, 7, 17, 22]) == 8

# Bài 2
P = [0]
for x in [2, 3, 1, 6, 7]:
    P.append(P[-1] ^ x)
pairs = [(x, y) for x in range(len(P)) for y in range(x + 1, len(P)) if P[x] == P[y]]
print("Bài 2: P =", P, "| cặp bằng nhau:", pairs)
assert P == [0, 2, 1, 0, 6, 1] and pairs == [(0, 3), (2, 5)]

# Bài 3
def count_todo_sol(arr):
    cnt, sumidx = defaultdict(int), defaultdict(int)
    cnt[0] = 1
    pre = ans = 0
    for y in range(1, len(arr) + 1):
        pre ^= arr[y - 1]
        ans += cnt[pre] * (y - 1) - sumidx[pre]
        cnt[pre] += 1
        sumidx[pre] += y
    return ans

for arr, want in examples:
    assert count_todo_sol(arr) == want
print("Bài 3: OK")

Bài 1: 8
Bài 2: P = [0, 2, 1, 0, 6, 1] | cặp bằng nhau: [(0, 3), (2, 5)]
Bài 3: OK


In [28]:
# Bài 4
def count_xor_k(arr, K):
    cnt = defaultdict(int)
    cnt[0] = 1
    pre = ans = 0
    for x in arr:
        pre ^= x
        ans += cnt[pre ^ K]
        cnt[pre] += 1
    return ans

def count_xor_k_brute(arr, K):
    n = len(arr)
    return sum(1 for i in range(n) for k in range(i, n) if reduce(xor, arr[i:k + 1], 0) == K)

# Bài 5
def count_triplets_k(arr, K):
    cnt, sumidx = defaultdict(int), defaultdict(int)
    cnt[0] = 1
    pre = ans = 0
    for y in range(1, len(arr) + 1):
        pre ^= arr[y - 1]
        ans += cnt[pre ^ K] * (y - 1) - sumidx[pre ^ K]
        cnt[pre] += 1
        sumidx[pre] += y
    return ans

def count_triplets_k_brute(arr, K):
    n, c = len(arr), 0
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j, n):
                if reduce(xor, arr[i:j], 0) ^ reduce(xor, arr[j:k + 1], 0) == K:
                    c += 1
    return c

rng = random.Random(19)
for _ in range(300):
    arr = [rng.randint(1, 7) for _ in range(rng.randint(1, 11))]
    K = rng.randint(0, 7)
    assert count_xor_k(arr, K) == count_xor_k_brute(arr, K)
    assert count_triplets_k(arr, K) == count_triplets_k_brute(arr, K)
    assert count_triplets_k(arr, 0) == count_linear(arr)
print("Bài 4, 5: OK (300 test ngẫu nhiên)")

Bài 4, 5: OK (300 test ngẫu nhiên)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template prefix XOR + dict (đếm và tổng chỉ số):**

```python
cnt, sumidx = defaultdict(int), defaultdict(int)
cnt[0] = 1                       # P[0] = 0 ở chỉ số 0
pre = ans = 0
for y in range(1, n + 1):
    pre ^= arr[y - 1]            # P[y]
    ans += cnt[pre] * (y - 1) - sumidx[pre]
    cnt[pre] += 1
    sumidx[pre] += y
```

**Template `O(n²)` dễ nhớ:**

```python
for i in range(n):
    cur = 0
    for k in range(i, n):
        cur ^= arr[k]
        if cur == 0: ans += k - i
```

**Checklist:**
- [ ] Đã dùng `a == b ⇔ xor(i..k) == 0`.
- [ ] Cộng `k − i` (không phải 1) cho mỗi đoạn XOR 0.
- [ ] Prefix có phần tử `P[0] = 0` (`cnt[0] = 1`).
- [ ] Cộng trước, ghi nhận sau trong vòng dict.
- [ ] Đã thử `[1,1]`, `[5]`, toàn số bằng nhau.